# Workshop — Agentes com dados estruturados, não estruturados e memória (Lakebase + Genie + LangGraph)

Dados **fictícios** para fins de treinamento. Nenhuma informação real de empresa ou pessoa é utilizada.

Neste workshop você vai montar, passo a passo, um **assistente regulatório** que responde perguntas combinando:

- **Dados estruturados** em tabelas do Unity Catalog, consultados em linguagem natural pelo **Genie**.
- **Dados não estruturados** de um relatório em PDF, processados com **AI Functions** (`ai_parse_document` e `ai_query` para embeddings).
- **Lakebase** (Postgres com pgvector) como armazenamento de vetores, **sessões** e **memória** de longo prazo.
- Um **agente LangGraph** com o modelo **Claude Sonnet** (system.ai), que usa o Genie como ferramenta.
- Um **Databricks App** (chat no estilo ChatGPT, com conversas anteriores na barra lateral e gráficos do Genie), com autorização de usuário (**OBO**).


## Pré-requisitos

- Workspace Databricks com **Unity Catalog**, **SQL Warehouse** (serverless recomendado), **AI Gateway** e **Lakebase Autoscaling** habilitado.
- Modelos do catálogo **system.ai** (via AI Gateway / `ai_query`): `system.ai.claude-sonnet-5-5` (chat) e `system.ai.qwen3-embedding-0-6b` (embeddings, 1024 dimensões).
- Schema `{catalogo}.{schema}` já existente, com permissão para criar tabelas nele.
- Volume para os PDFs já existente (criado pelo instrutor). O path completo entra na variável `VOLUME_PDF_PATH` (Bloco 1).
- Arquivos do repositório: `dados/*.csv`, `documentos/relatorio_regulatorio_ficticio_2025.pdf` e a pasta `app/`.

---
## Bloco 0 — Dependências

Instala as bibliotecas Python usadas nas Seções 4 e 7. Ao final, o Python é reiniciado.

In [ ]:
%pip install -q "psycopg[binary]>=3.2" "databricks-sdk>=0.89.0"
dbutils.library.restartPython()

---
## Bloco 1 — Configuração

Ajuste as variáveis da célula abaixo com os valores do seu ambiente:

| Variável | O que colocar | Exemplo |
|---|---|---|
| `CATALOGO` | Catálogo do workshop | `workspace` |
| `SCHEMA` | Schema para as tabelas | `workshop_agentes` |
| `VOLUME_PDF_PATH` | Path completo do volume dos PDFs | `/Volumes/workspace/workshop_agentes/docs_regulatorios` |
| `PREFIXO` | Identificador único em minúsculas (sem underline). Usado no nome das tabelas (`<prefixo>_<tabela>`), do projeto Lakebase e do app | `ana` |

O `APP_ID` (Client ID do service principal do app) é preenchido depois, na Seção 7.5.

In [ ]:
CATALOGO = "workspace"
SCHEMA = "workshop_agentes"
VOLUME_PDF_PATH = "/Volumes/workspace/workshop_agentes/docs_regulatorios"  # path completo do volume dos PDFs
PREFIXO = "ana"  # seu prefixo: minúsculas, sem underline

VOLUME_PDF_PATH = VOLUME_PDF_PATH.rstrip("/")
PREFIXO = PREFIXO.strip().rstrip("_")

PREFIXO_SLUG = PREFIXO.lower().replace("_", "-").strip("-")  # nomes de projeto e app aceitam só letras, números e hífen
PROJETO_LAKEBASE = f"workshop-agentes-{PREFIXO_SLUG}"
MODELO_CHAT = "system.ai.claude-sonnet-5-5"          # AI Gateway / Unity Catalog
MODELO_EMBEDDING = "system.ai.qwen3-embedding-0-6b"  # AI Gateway / Unity Catalog

print(f"Catálogo/Schema: {CATALOGO}.{SCHEMA}")
print(f"Tabelas UC:      {PREFIXO}_indicadores_operacionais, {PREFIXO}_tarifas_segmento, {PREFIXO}_documentos_parseados, {PREFIXO}_documentos_chunks")
print(f"Modelo chat:     {MODELO_CHAT}")
print(f"Modelo embed:    {MODELO_EMBEDDING}")
print(f"PDFs em:         {VOLUME_PDF_PATH}")
print(f"Projeto Lakebase: {PROJETO_LAKEBASE}")
print(f"Nome do app:     assistente-regulatorio-{PREFIXO_SLUG}")

---
## Bloco 2 — Dados estruturados: CSV no Unity Catalog

O workshop usa **dois arquivos CSV** da pasta `dados/` do repositório. Eles são carregados **manualmente** pela interface do Databricks, como tabelas do Unity Catalog:

| Arquivo | Tabela no Unity Catalog | Conteúdo |
|---|---|---|
| `dados/indicadores_operacionais.csv` | `{prefixo}_indicadores_operacionais` | Indicadores mensais por região (2022 a 2026, até setembro): volume, clientes, rede, perdas, emergência, vazamentos e investimento |
| `dados/tarifas_segmento.csv` | `{prefixo}_tarifas_segmento` | Tarifas trimestrais por segmento e faixa de consumo (margem, molécula, transporte, impostos e tarifa final) |

> **Importante:** para que os participantes não sobrescrevam as tabelas uns dos outros no mesmo schema, **sempre adicione o seu prefixo seguido de underline ao nome da tabela** (por exemplo, se o prefixo for `ana`, use `ana_indicadores_operacionais`).

### 2.1 — Upload do primeiro CSV (`indicadores_operacionais`)

1. No menu lateral, abra **Catalog**.
2. Clique em **+ (Add data)** → **Create table** (ou **Create or modify table**, conforme a versão da interface).
3. Em **Upload file**, arraste `dados/indicadores_operacionais.csv` ou clique em **browse** e selecione o arquivo.
4. Em **Catalog** e **Schema**, escolha `{catalogo}` → `{schema}`.
5. **Nome da tabela:** adicione o seu prefixo seguido de underline ao nome: **`{prefixo}_indicadores_operacionais`** (por exemplo: `ana_indicadores_operacionais`).
6. Em **Advanced attributes**, mantenha **First row is header** marcado. Confira os tipos das colunas (inferidos pelo Databricks) e clique em **Create table**.

### 2.2 — Upload do segundo CSV (`tarifas_segmento`)

Repita o mesmo caminho com `dados/tarifas_segmento.csv`:

1. **Catalog** → **+ (Add data)** → **Create table**.
2. Arquivo: `dados/tarifas_segmento.csv`.
3. Schema: `{catalogo}` → `{schema}`.
4. **Nome da tabela:** adicione o seu prefixo seguido de underline ao nome: **`{prefixo}_tarifas_segmento`** (por exemplo: `ana_tarifas_segmento`).
5. Mantenha **First row is header** marcado e clique em **Create table**.

> Se a tabela já existir (por exemplo, de uma tentativa anterior), exclua-a em **Catalog** antes de criar de novo, ou escolha outro nome.

---
## Bloco 3 — Dados não estruturados: PDF com AI Functions

O relatório regulatório (10 páginas, com textos, tabelas e gráficos em imagem) será tratado com **AI Functions** em SQL:

1. **`ai_parse_document`** extrai o conteúdo do PDF (textos, títulos, tabelas e legendas) em um objeto `VARIANT`.
2. Os elementos são agrupados **por página** formando os *chunks*.
3. **`ai_query`** com o modelo de embeddings do **system.ai** (AI Gateway) gera um vetor de 1024 dimensões para cada chunk.

### 3.1 — Upload manual do PDF

1. Confira a variável **`VOLUME_PDF_PATH`** no Bloco 1 e ajuste-a com o path completo do volume (exemplo: `/Volumes/workspace/workshop_agentes/docs_regulatorios`).
2. No **Catalog**, abra esse volume e clique em **Upload to this volume**.
3. Envie o arquivo `relatorio_regulatorio_ficticio_2025.pdf` para a **raiz** do volume (sem criar subpastas).

In [ ]:
display(dbutils.fs.ls(VOLUME_PDF_PATH))

### 3.2 — Parse do PDF com `ai_parse_document`

Cada linha representa um documento. A coluna `parsed` guarda a estrutura completa (elementos com tipo, conteúdo, página e posição).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.{SCHEMA}.{PREFIXO}_documentos_parseados AS
SELECT regexp_extract(path, '[^/]+$', 0) AS documento,
       ai_parse_document(content) AS parsed
FROM read_files('{VOLUME_PDF_PATH}/*.pdf', format => 'binaryFile')
""")

In [ ]:
# Veja o primeiro elemento extraído e o tipo de cada elemento
display(spark.sql(f"""
SELECT documento,
       to_json(parsed:document:elements[0]) AS primeiro_elemento
FROM {CATALOGO}.{SCHEMA}.{PREFIXO}_documentos_parseados
"""))

### 3.3 — Chunks por página e embeddings com `ai_query`

Os elementos de cada página são unidos (na ordem do documento) em um chunk. Depois, `ai_query` gera o embedding de cada chunk com o modelo **`system.ai.qwen3-embedding-0-6b`** (AI Gateway).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.{SCHEMA}.{PREFIXO}_documentos_chunks AS
WITH elementos AS (
  SELECT documento, e.id AS id, e.content AS conteudo, e.bbox[0].page_id AS page_id
  FROM (
    SELECT documento,
           explode(from_json(to_json(parsed:document:elements),
             'ARRAY<STRUCT<id:INT,content:STRING,bbox:ARRAY<STRUCT<page_id:INT>>>>')) AS e
    FROM {CATALOGO}.{SCHEMA}.{PREFIXO}_documentos_parseados
  )
  WHERE e.content IS NOT NULL AND e.bbox[0].page_id IS NOT NULL
),
paginas AS (
  SELECT documento,
         page_id + 1 AS pagina,
         array_join(transform(sort_array(collect_list(struct(id, conteudo))), x -> x.conteudo), '\n') AS conteudo
  FROM elementos
  GROUP BY documento, page_id
)
SELECT documento,
       pagina,
       conteudo,
       ai_query('{MODELO_EMBEDDING}', conteudo) AS embedding
FROM paginas
ORDER BY pagina
""")

### 3.4 — Validar os chunks

Esperado: **10** linhas (uma por página) e embeddings de tamanho **1024**.

In [ ]:
display(spark.sql(f"""
SELECT pagina, LEFT(conteudo, 120) AS trecho, size(embedding) AS dimensoes
FROM {CATALOGO}.{SCHEMA}.{PREFIXO}_documentos_chunks
ORDER BY pagina
"""))

### 3.5 — Claude Sonnet via `ai_query` (system.ai / AI Gateway)

O mesmo modelo de chat que o agente vai usar (`system.ai.claude-sonnet-5-5`). Aqui ele resume a página 9 (ocorrências relevantes).

In [ ]:
display(spark.sql(f"""
SELECT ai_query(
  '{MODELO_CHAT}',
  CONCAT('Resuma em três tópicos curtos o trecho a seguir de um relatório regulatório: ', conteudo)
) AS resumo_pagina_9
FROM {CATALOGO}.{SCHEMA}.{PREFIXO}_documentos_chunks
WHERE pagina = 9
"""))

---
## Bloco 4 — Lakebase: vetores, sessões e memória

O **Lakebase** é o Postgres gerenciado do Databricks. Neste workshop ele guarda três coisas:

| Tabela | Para que serve |
|---|---|
| `documentos_chunks` | Trechos do relatório com embedding **pgvector** (busca por similaridade) |
| `sessoes` | Lista de conversas do usuário (título, criação e atualização) |
| `memorias_usuario` | Memórias de longo prazo do usuário (preferências e fatos pedidos para lembrar), com embedding |

O histórico de cada conversa (checkpoints do LangGraph) é criado automaticamente pelo app na primeira execução.

### 4.1 — Criar o projeto Lakebase

1. No menu lateral, abra **Compute → Lakebase Postgres** (ou pesquise por *Lakebase*).
2. Clique em **New project**.
3. **Project name:** `workshop-agentes-<seu prefixo>` (exatamente o valor de `PROJETO_LAKEBASE` impresso no Bloco 1).
4. **Postgres version:** `17`.
5. **Capacity / Scaling:** selecione **1–2 ACUs** (Autoscaling Compute Units). Essa capacidade mínima é suficiente para o workshop e economiza recursos.
6. Mantenha o branch padrão `production` e o endpoint `primary`.
7. Clique em **Create** e aguarde o status **Ready**.

### 4.2 — Conectar ao Lakebase a partir do notebook

A conexão usa um **token OAuth** de curta duração (~60 min) gerado pelo SDK, com a identidade de quem executa o notebook. Não há senha no código.

In [ ]:
import psycopg
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ramo = f"projects/{PROJETO_LAKEBASE}/branches/production"
endpoint = next(iter(w.postgres.list_endpoints(parent=ramo)))
ENDPOINT_NAME = endpoint.name
HOST = endpoint.status.hosts.host
USUARIO = w.current_user.me().user_name

def conectar_lakebase():
    token = w.postgres.generate_database_credential(endpoint=ENDPOINT_NAME).token
    return psycopg.connect(
        host=HOST,
        dbname="databricks_postgres",
        user=USUARIO,
        password=token,
        sslmode="require",
        autocommit=True,
    )

print("Endpoint:", ENDPOINT_NAME)
print("Usuário:", USUARIO)

### 4.3 — Criar extensão `vector` e tabelas

As tabelas `sessoes` e `memorias_usuario` são criadas aqui para que as permissões da role do app (Seção 7) sejam aplicadas sobre elas.

In [ ]:
with conectar_lakebase() as conn:
    conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.documentos_chunks (
            id        SERIAL PRIMARY KEY,
            documento TEXT NOT NULL,
            pagina    INT  NOT NULL,
            conteudo  TEXT NOT NULL,
            embedding vector(1024) NOT NULL
        )""")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.sessoes (
            session_id    UUID PRIMARY KEY,
            user_email    TEXT NOT NULL,
            titulo        TEXT NOT NULL,
            criado_em     TIMESTAMPTZ NOT NULL DEFAULT now(),
            atualizado_em TIMESTAMPTZ NOT NULL DEFAULT now()
        )""")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.memorias_usuario (
            id         SERIAL PRIMARY KEY,
            user_email TEXT NOT NULL,
            memoria    TEXT NOT NULL,
            embedding  vector(1024) NOT NULL,
            criado_em  TIMESTAMPTZ NOT NULL DEFAULT now()
        )""")
    print("Extensão e tabelas prontas.")

### 4.4 — Carregar os chunks do Unity Catalog para o Lakebase

Os embeddings gerados na Seção 3 são copiados para a tabela `documentos_chunks` do Lakebase. Assim o agente faz a busca vetorial no próprio Postgres.

In [ ]:
linhas = spark.table(f"{CATALOGO}.{SCHEMA}.{PREFIXO}_documentos_chunks").collect()

with conectar_lakebase() as conn:
    conn.execute("TRUNCATE public.documentos_chunks RESTART IDENTITY")
    for r in linhas:
        vetor = "[" + ",".join(str(x) for x in r["embedding"]) + "]"
        conn.execute(
            "INSERT INTO public.documentos_chunks (documento, pagina, conteudo, embedding) "
            "VALUES (%s, %s, %s, %s::vector)",
            (r["documento"], int(r["pagina"]), r["conteudo"], vetor),
        )
    total = conn.execute("SELECT count(*) FROM public.documentos_chunks").fetchone()[0]

print(f"Chunks no Lakebase: {total}")

### 4.5 — Testar a busca vetorial

A pergunta é convertida em embedding com o mesmo modelo do **system.ai** (via `ai_query` / AI Gateway) e comparada com os chunks por distância de cosseno (`<=>`).

In [ ]:
pergunta = "Qual era a meta regulatória de perdas de gás para 2025?"
pergunta_sql = pergunta.replace("'", "''")
emb = spark.sql(
    f"SELECT ai_query('{MODELO_EMBEDDING}', '{pergunta_sql}') AS embedding"
).collect()[0]["embedding"]
vetor = "[" + ",".join(str(x) for x in emb) + "]"

with conectar_lakebase() as conn:
    resultados = conn.execute(
        """SELECT pagina, LEFT(conteudo, 160), 1 - (embedding <=> %s::vector) AS similaridade
           FROM public.documentos_chunks
           ORDER BY embedding <=> %s::vector
           LIMIT 3""",
        (vetor, vetor),
    ).fetchall()

for pagina, trecho, sim in resultados:
    print(f"p.{pagina} | similaridade {sim:.3f} | {trecho.replace(chr(10), ' ')}...")

---
## Bloco 5 — Genie Agent: a ferramenta de dados

O **Genie Space** responde perguntas em linguagem natural gerando SQL sobre as tabelas do Unity Catalog. No agente, ele aparece como a ferramenta `consultar_dados`: o agente envia a pergunta, o Genie executa a consulta e devolve a resposta com o SQL usado.

### 5.1 — Criar o Genie Space (passo a passo)

1. No menu lateral, abra **Genie** (ou **Genie Spaces**) e clique em **New**.
2. **Título:** `Indicadores Regulatórios - <seu prefixo>`.
3. **Description (descrição da sala):** preencha o campo com o texto abaixo.

```
Assistente de análise de indicadores operacionais e de tarifas de uma distribuidora de gás natural. Responde sobre perdas de gás (realizado x meta), tempo médio de atendimento a emergências, ocorrências de vazamento, volume distribuído, clientes ativos, extensão de rede e investimentos por região (Capital, Litoral, Planalto Norte, Vale do Rio Claro e Zona Industrial Sul), com dados mensais de 2022 a setembro de 2026. Também cobre as tarifas trimestrais por segmento (Residencial, Comercial, Industrial, GNV e Cogeração) e faixa de consumo, com a composição do preço (margem de distribuição, molécula, transporte e impostos) e os reajustes trimestrais. Use para comparar regiões e segmentos, montar rankings, identificar tendências e calcular médias anuais.
```

   > **Como adicionar a descrição na sala:**
   > 1. Abra a sala do Genie que você criou (**Genie → sua sala**).
   > 2. Clique no ícone de **Settings** (engrenagem) ou em **Edit** no cabeçalho da sala.
   > 3. No campo **Description**, cole o texto acima.
   > 4. Clique em **Save**. A descrição aparece para os usuários que acessam a sala.
   >
   > Os nomes dos botões podem variar um pouco conforme a versão da interface.

4. **SQL warehouse:** selecione o warehouse do workshop.
5. **Tabelas:** clique em **Add tables** e escolha `{prefixo}_indicadores_operacionais` e `{prefixo}_tarifas_segmento` do schema `{schema}`.
6. Em **Instructions** (ou *Settings → Instructions*), cole o texto abaixo (ajustando o nome das tabelas para o seu prefixo):

```
Você é um analista de dados de uma distribuidora fictícia de gás natural. Responda sempre em português.

Tabelas:
- <seu prefixo>_indicadores_operacionais: indicadores mensais por região, de 2022 a setembro de 2026. Colunas: volume distribuído (mil m³), clientes ativos, extensão de rede (km), perdas_pct (perdas, em %), meta_perdas_pct (meta, em %), tempo_medio_emergencia_min (minutos), ocorrencias_vazamento e investimento_realizado_mi_rs (R$ milhões).
- <seu prefixo>_tarifas_segmento: tarifas trimestrais por segmento e faixa de consumo. Colunas: margem_distribuicao_rs_m3, molecula_rs_m3, transporte_rs_m3, impostos_pct, tarifa_final_rs_m3 (R$/m³) e reajuste_pct_vs_trimestre_anterior.

Regras:
- Para médias anuais, use a média simples dos meses do ano.
- Sempre informe a unidade de medida e o critério usado (média simples, soma, variação percentual).
- Em perguntas hipotéticas, deixe as premissas explícitas.
- Se a pergunta não puder ser respondida com estas tabelas, diga isso claramente.
```

7. Em **Sample questions** (perguntas sugeridas), adicione 3 a 5 perguntas da lista 5.2.
8. Clique em **Save** e teste algumas perguntas na própria tela do Genie.
9. **Compartilhe** o Genie Space com os participantes (permissão **Can Run**).
10. Copie o **ID do Genie Space** da URL (`.../genie/rooms/<ID>`) e guarde para o `app.yaml` (Seção 7).

### 5.2 — Perguntas para o Genie Agent

Perguntas de **análise**:

1. Qual foi a média de perdas de gás (`perdas_pct`) por ano? Mostre em tabela.
2. Qual região teve o maior tempo médio de emergência em 2025 e qual a diferença para a média das regiões?
3. Em 2025, quais regiões ficaram acima da meta de perdas (`meta_perdas_pct`)?
4. Quantas ocorrências de vazamento houve em julho de 2024 no Litoral, comparadas com a média dos outros meses de 2024?
5. Como o investimento realizado evoluiu por ano? Compare com o volume distribuído anual.
6. Como a tarifa final residencial (Faixa 1) evoluiu entre 2022 e o 3º trimestre de 2026? Calcule a variação percentual.

Perguntas **hipotéticas** (o Genie Agent deve declarar as premissas):

7. Se a região Planalto Norte reduzisse as perdas para a meta de 2025 em 2026, quanto volume (mil m³) deixaria de ser perdido, considerando o volume distribuído de 2025?
8. Se o tempo médio de emergência de todas as regiões caísse 10% em 2026, qual seria a média geral?

---
## Bloco 6 — Genie One: exploração e report agendado

O **Genie One** é a experiência de Genie voltada ao negócio: perguntas em linguagem natural, respostas com gráficos e a possibilidade de agendar um report recorrente. Ele usa os mesmos Genie Spaces criados na Seção 5.

> As telas podem variar um pouco conforme a versão do workspace. Os nomes exatos dos menus estão em inglês/português conforme a interface.

### 6.1 — Passo a passo para usar o Genie One

1. No menu do workspace, abra **Genie One** (ou **Databricks One**, se disponível).
2. Em **Spaces** (ou *Genie spaces*), selecione **`Indicadores Regulatórios - <seu prefixo>`**.
3. Digite a pergunta na caixa de mensagem e aguarde a resposta.
4. Abra **Show code** (ou *Ver SQL*) para conferir a consulta gerada.
5. Use o ícone de gráfico para transformar a tabela em visualização e clique em **Add to dashboard** se quiser guardar a análise.
6. Para agendar um report: clique no resultado (ou na conversa) → **Schedule** (ou *Agendar*) → defina a frequência (diária, às 08:00) → destinatários → **Save**.

### 6.2 — Perguntas para o Genie One

1. Quanto volume foi distribuído em 2025, por região?
2. Como as perdas de gás evoluíram ano a ano?
3. Monte um resumo executivo com os três principais destaques de 2025.
4. Quais regiões precisam de atenção no tempo de atendimento de emergência?
5. Mostre a tarifa final residencial por trimestre em 2026, em gráfico de linhas.
6. **Agende um report diário** com a média de perdas por região e envie para você mesmo.
7. **Crie uma história no Canvas** com volume por região, perdas ano a ano e a tarifa residencial em 2026; depois **exporte a história em PDF**.

---
## Bloco 7 — Agente LangGraph + Databricks App com OBO

### 7.1 — Como o agente funciona

O código está na pasta `app/` do repositório:

| Arquivo | Função |
|---|---|
| `app/agente.py` | Agente **LangGraph** (`create_react_agent`) com o modelo do **system.ai** (via AI Gateway), quatro ferramentas e o checkpointer do Lakebase |
| `app/lakebase.py` | Conexão com o Lakebase (token OAuth do app), sessões, memória e busca de documentos |
| `app/app.py` | Interface Streamlit no estilo ChatGPT: **Nova conversa**, conversas anteriores na barra lateral, sugestões de perguntas e gráficos/tabelas do Genie |
| `app/graficos.py` | Redesenha (Altair) os gráficos que o Genie gera, a partir dos dados da consulta |
| `app/.streamlit/config.toml` | Tema visual do app |
| `app/app.yaml` | Configuração do app (variáveis e escopos OBO). Os valores marcados com `xxxxxxx` devem ser preenchidos |
| `app/requirements.txt` | Dependências do app |

As ferramentas do agente:

| Ferramenta | Onde os dados ficam | Quem autoriza |
|---|---|---|
| `buscar_documentos` | Lakebase (`documentos_chunks`, pgvector) | Service principal do app |
| `consultar_dados` | Genie Space (Unity Catalog) | **Usuário** (OBO), com o token do próprio usuário |
| `salvar_memoria` / `buscar_memorias` | Lakebase (`memorias_usuario`) | Service principal, isolado por `user_email` |

As sessões são guardadas pelo **checkpointer** do LangGraph no Lakebase, com o `thread_id` = `email:session_id`. Por isso, ao selecionar uma sessão anterior, o histórico é recuperado.

### 7.2 — Preparar o `app.yaml`

Abra `app/app.yaml` e substitua apenas os valores marcados:

| Variável | O que colocar |
|---|---|
| `GENIE_SPACE_ID` | ID do Genie Space (Seção 5.1, passo 9) |
| `MODEL_ENDPOINT` | Mantenha `system.ai.claude-sonnet-5-5` (ou o nome do modelo de chat do seu workspace) |
| `EMBEDDING_ENDPOINT` | Mantenha `system.ai.qwen3-embedding-0-6b` |
| `DOCUMENTOS_TABELA` | Mantenha `documentos_chunks` |
| `LAKEBASE_ENDPOINT` | Mantenha `valueFrom: postgres` (o valor é injetado pelo app) |

### 7.3 — Criar o app

1. No menu lateral, abra **Compute → Apps** e clique em **Create app**.
2. Nome: **`assistente-regulatorio-<seu prefixo>`** (com `-` no lugar de `_`, se o prefixo tiver `_`).
3. Clique em **Create** e **aguarde 2 a 3 minutos** até o app ser criado. Nessa etapa ele ainda não foi implantado.
4. Na página do app, clique em **Deploy** e, em **Source code path**, **selecione no workspace a pasta `app/`** do repositório (por exemplo, `/Workspace/Users/<seu-email>/databricks-workshop-agents-custom/app`).
5. Confirme o deploy e aguarde o status **Running**.

> A pasta `app/` precisa estar no workspace antes do passo 4. Se o repositório ainda não estiver lá, importe-o (ou use a CLI, na opção abaixo).

> Opção via CLI: `databricks apps create assistente-regulatorio-<prefixo>`, depois `databricks workspace import-dir ./app /Workspace/Users/<seu-email>/apps/assistente-regulatorio --overwrite` e `databricks apps deploy assistente-regulatorio-<prefixo> --source-code-path /Workspace/Users/<seu-email>/apps/assistente-regulatorio`.

### 7.4 — Anexar o Lakebase como recurso do app

1. Na página do app, abra **Settings → Resources** (ou *Edit*) e clique em **+ Add resource**.
2. Tipo **Database** → **Lakebase Autoscaling**.
3. Selecione o projeto `workshop-agentes-<seu prefixo>`, o branch `production` e o banco `databricks_postgres`.
4. Permissão: **Can connect and create**.
5. **Resource key:** `postgres`. O nome precisa ser exatamente este, porque o `app.yaml` usa `valueFrom: postgres` para ler o Lakebase.
   > **Atenção:** se o recurso foi criado com outro nome (por exemplo, `database`), o app não recebe `LAKEBASE_ENDPOINT` nem `PGHOST` e falha ao conectar, com a mensagem `Detalhe: 'LAKEBASE_ENDPOINT'`. Para corrigir, edite o recurso, troque a **Resource key** para `postgres` e faça **Deploy** de novo.
6. Salve. O runtime passa a injetar `PGHOST`, `PGUSER`, `PGDATABASE`, `PGPORT` e `PGSSLMODE`.
7. Depois de salvar, faça **Deploy** novamente para o app receber as variáveis.

### 7.5 — Permissões do service principal do app

1. Copie o **Client ID** (App ID) do service principal, na página de detalhes do app, e cole na variável **`APP_ID`** da célula abaixo.
2. Execute a célula abaixo. Ela gera o SQL que dá acesso ao Lakebase para a role do app.

In [ ]:
APP_ID = ""  # cole aqui o Client ID do service principal (página de detalhes do app)
APP_ID = APP_ID.strip()
if not APP_ID:
    print("Preencha a variável APP_ID com o Client ID do service principal e rode novamente.")
else:
    print(f"""-- Execute no SQL Editor do projeto Lakebase (banco databricks_postgres)

CREATE EXTENSION IF NOT EXISTS databricks_auth;

-- Ao anexar o Lakebase como recurso do app (7.4), a plataforma normalmente já cria a role do app.
-- Este bloco só cria a role OAuth se ela ainda não existir.
DO $$
BEGIN
   IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = '{APP_ID}') THEN
      PERFORM databricks_create_role('{APP_ID}', 'SERVICE_PRINCIPAL');
   END IF;
END
$$;

GRANT USAGE, CREATE ON SCHEMA public TO \"{APP_ID}\";
GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA public TO \"{APP_ID}\";
GRANT USAGE, SELECT, UPDATE ON ALL SEQUENCES IN SCHEMA public TO \"{APP_ID}\";
""")

3. Cole o SQL gerado no **SQL Editor** do projeto Lakebase e execute.

> Não use `CREATE ROLE` comum para o service principal: essa role não fica vinculada à identidade Databricks e a autenticação do app falha com `password authentication failed`. A role OAuth é criada por `databricks_create_role` (ou automaticamente pela plataforma ao anexar o recurso).
>
> Se aparecer `permission denied to drop objects`, não há problema: significa que a role já existe e foi criada pela plataforma. Não é preciso removê-la.

4. **Dados (Unity Catalog):** o Genie consulta os dados com a permissão de cada usuário (OBO). Conceda aos participantes:

```sql
-- Troque pelos valores de CATALOGO e SCHEMA do Bloco 1
GRANT USE CATALOG ON CATALOG workspace TO `<grupo-ou-email-dos-participantes>`;
GRANT USE SCHEMA ON SCHEMA workspace.workshop_agentes TO `<grupo-ou-email-dos-participantes>`;
GRANT SELECT ON SCHEMA workspace.workshop_agentes TO `<grupo-ou-email-dos-participantes>`;
```

### 7.6 — Autorização de usuário (OBO)

O app pede ao usuário a autorização para as APIs listadas em `user_api_scopes` do `app.yaml`. Na página do app, abra **Edit** (ou **Settings**) → **User authorization** e adicione estes três escopos (os mesmos do `app.yaml`):

| Scope | Para que serve |
|---|---|
| `sql` | Executar SQL e gerenciar recursos SQL no Databricks (consultas do Genie no warehouse). |
| `genie` | Acessar o Databricks Genie como o próprio usuário. |
| `ai-gateway` | Rotear requisições de inferência pelo Unity Gateway (modelos do `system.ai`). |

1. Clique em **+ Add scope**, selecione `sql`, `genie` e `ai-gateway`, e clique em **Save**.
   > O valor de `user_api_scopes` no `app.yaml` é a referência do workshop. Os escopos precisam estar **configurados no app** para o token do usuário ser entregue ao código.
2. **Configure os escopos antes de abrir a URL do app pela primeira vez.** Na primeira abertura, o Databricks mostra a tela de consentimento com os escopos configurados e guarda o que o usuário aprovou.
3. Abra a URL do app e **aprove todos os escopos** (`sql`, `genie` e `ai-gateway`) na tela de consentimento.

> **Se aparecer `403 Forbidden · Invalid scope, required scopes: genie`:** o consentimento foi dado antes de os escopos serem adicionados e ficou só com os escopos padrão. Mesmo com `app.yaml` e app corretos, o token chega sem `genie`. Configure os escopos no app, abra a URL em uma janela anônima (ou saia e entre de novo) e aprove todos os escopos na tela de consentimento.

O token chega ao app no header `x-forwarded-access-token`. Sem o escopo `genie` nesse token, a ferramenta `consultar_dados` falha.

### 7.7 — Testar o app

1. Abra a URL do app.
2. Clique em **Nova conversa** (ou em uma das sugestões da tela inicial) e faça as perguntas da lista 7.8.
3. Peça um resultado em gráfico (ex.: "Mostre em um gráfico o volume distribuído por região em 2025"): o gráfico do Genie aparece na resposta, com **Ver dados e SQL**.
4. Recarregue a página e clique em uma conversa da barra lateral: o histórico (inclusive os gráficos) deve voltar.
5. Peça ao agente para lembrar uma preferência, inicie outra conversa e pergunte sobre ela.

### 7.8 — Perguntas para o agente customizado

Perguntas que combinam **relatório (documentos)** e **Genie (dados)**, com contexto entre turnos:

1. Qual foi a média de perdas de gás em 2025 e qual era a meta regulatória? Responda com os números.
2. E qual região teve o maior tempo médio de emergência em 2025?
3. O que o relatório diz sobre o evento climático no Litoral em julho de 2024? Cite a página.
4. Quais ações corretivas foram tomadas depois desse evento?
5. Quais compromissos a distribuidora assumiu para o próximo ciclo?
6. Qual é a tarifa final residencial (Faixa 1) no 3º trimestre de 2026?

Perguntas de **memória e sessões**:

7. *(Sessão 1)* Lembre que eu prefiro respostas com tabelas sempre que houver números.
8. *(Sessão 2, nova)* Quais são as minhas preferências de formato de resposta?
9. *(Sessão 1, recuperada pela barra lateral)* Resuma o que conversamos até agora.

Pergunta **fora do escopo** (o agente deve dizer que não encontrou a informação nas fontes):

10. Qual é a cotação do dólar hoje?